# SSG 랜더스 (구 SK 와이번스) 분석

팀 성적 트렌드 + 주요 선수 커리어 트래킹

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.font_manager as fm
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# 한글 폰트
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc']:
    try:
        fm.fontManager.addfont(fp)
        plt.rcParams['font.family'] = fm.FontProperties(fname=fp).get_name()
        break
    except: pass
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

RAW_DIR = Path('../data/raw/kbo_official')
SSG_COLOR  = '#C8102E'  # SSG 랜더스 레드
SSG_COLOR2 = '#003087'  # SSG 네이비

In [ ]:
# ── 데이터 로드 ────────────────────────────────────────────────────────────────

def load_all() -> pd.DataFrame:
    files = sorted(RAW_DIR.glob('kbo_*.csv'))
    if not files:
        raise FileNotFoundError('데이터 없음. src/crawl_kbo.py 먼저 실행하세요.')
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    num_skip = {'선수명', '팀명'}
    for c in df.columns:
        if c not in num_skip:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    # SK / SSG 통일
    df['팀명_통일'] = df['팀명'].replace({'SK': 'SSG', 'sk': 'SSG'})
    return df

all_df = load_all()
ssg_df = all_df[all_df['팀명'].isin(['SK', 'SSG'])].copy()

print(f'전체 {len(all_df)}행 | SSG/SK {len(ssg_df)}행')
print(f'수집 시즌: {sorted(ssg_df["Season"].unique())}')
print(f'컬럼: {list(ssg_df.columns)}')

In [ ]:
# ── 0. SSG 시즌별 로스터 확인 ─────────────────────────────────────────────────

def show_roster(season: int, min_pa: int = 50):
    d = ssg_df[(ssg_df['Season'] == season) & (ssg_df['PA'] >= min_pa)]
    show = ['선수명', 'PA', 'AVG', 'HR', 'RBI', 'SB', 'BB', 'SO']
    show = [c for c in show if c in d.columns]
    print(f'\n=== {season} SSG 타자 (PA≥{min_pa}) ===')
    print(d[show].sort_values('PA', ascending=False).to_string(index=False))

latest = int(ssg_df['Season'].max())
show_roster(latest)

In [ ]:
# ── 1. 팀 타선 종합 지표 연도별 추이 ──────────────────────────────────────────

def plot_team_trend():
    # PA 가중 평균으로 팀 대표값 계산
    def wavg(grp, col):
        valid = grp.dropna(subset=[col, 'PA'])
        return (valid[col] * valid['PA']).sum() / valid['PA'].sum() if not valid.empty else np.nan

    seasons = sorted(ssg_df['Season'].unique())
    metrics = {
        'AVG':    '타율',
        'HR':     '홈런 합계',
        'RBI':    '타점 합계',
        'BB':     '볼넷 합계',
        'SO':     '삼진 합계',
    }

    agg = {}
    for s in seasons:
        grp = ssg_df[ssg_df['Season'] == s]
        row = {'Season': s}
        row['AVG_wavg']  = wavg(grp, 'AVG')
        row['HR_sum']    = grp['HR'].sum()
        row['RBI_sum']   = grp['RBI'].sum()
        row['BB_sum']    = grp['BB'].sum()
        row['SO_sum']    = grp['SO'].sum()
        row['PA_sum']    = grp['PA'].sum()
        row['선수수']     = len(grp[grp['PA'] >= 100])
        agg[s] = row
    team = pd.DataFrame(agg.values())

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle('SSG 랜더스 (구 SK 와이번스) 시즌별 타선 지표', fontsize=15, fontweight='bold')

    pairs = [
        (axes[0,0], 'AVG_wavg',  'PA 가중 팀타율',  True),
        (axes[0,1], 'HR_sum',    '팀 홈런 합계',    False),
        (axes[0,2], 'RBI_sum',   '팀 타점 합계',    False),
        (axes[1,0], 'BB_sum',    '팀 볼넷 합계',    False),
        (axes[1,1], 'SO_sum',    '팀 삼진 합계',    False),
        (axes[1,2], '선수수',     '주전(PA≥100) 수',False),
    ]

    for ax, col, title, is_pct in pairs:
        vals = team[col]
        ax.fill_between(team['Season'], vals, alpha=0.15, color=SSG_COLOR)
        ax.plot(team['Season'], vals, marker='o', color=SSG_COLOR, linewidth=2.5, markersize=6)
        if is_pct:
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.3f}'))
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('시즌')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(alpha=0.3)
        # 최고/최저 표시
        if vals.notna().any():
            idx_max = vals.idxmax()
            ax.annotate(f'{vals[idx_max]:.0f}' if not is_pct else f'{vals[idx_max]:.3f}',
                        xy=(team['Season'][idx_max], vals[idx_max]),
                        xytext=(0, 8), textcoords='offset points',
                        ha='center', fontsize=8, color=SSG_COLOR, fontweight='bold')

    # SK→SSG 전환 표시
    for ax, *_ in pairs:
        ax.axvline(2021.5, color='gray', linestyle=':', alpha=0.7, linewidth=1)
        ax.text(2021.6, ax.get_ylim()[0], 'SK→SSG', fontsize=7, color='gray')

    plt.tight_layout()
    plt.savefig('../data/ssg_team_trend.png', bbox_inches='tight')
    plt.show()

plot_team_trend()

In [ ]:
# ── 2. SSG vs 리그 평균 비교 히트맵 ──────────────────────────────────────────

def plot_ssg_vs_league():
    def wavg_season(df, season, col):
        g = df[df['Season'] == season].dropna(subset=[col, 'PA'])
        return (g[col] * g['PA']).sum() / g['PA'].sum() if not g.empty else np.nan

    seasons = sorted(ssg_df['Season'].unique())
    metrics = ['AVG', 'HR', 'RBI', 'BB', 'SO']
    metrics = [m for m in metrics if m in ssg_df.columns]

    diff_data = []
    for s in seasons:
        row = {'Season': s}
        for m in metrics:
            ssg_val    = wavg_season(ssg_df, s, m)
            league_val = wavg_season(all_df, s, m)
            if m in ['HR', 'RBI', 'BB', 'SO']:
                # 합계 기반 비교: 팀 합계 / 리그 팀수
                ssg_sum    = ssg_df[ssg_df['Season']==s][m].sum()
                league_avg = all_df[all_df['Season']==s].groupby('팀명')[m].sum().mean()
                row[m] = ssg_sum - league_avg
            else:
                row[m] = (ssg_val - league_val) / league_val * 100 if league_val else np.nan
        diff_data.append(row)

    diff_df = pd.DataFrame(diff_data).set_index('Season')

    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.imshow(diff_df[metrics].T.values, aspect='auto', cmap='RdYlGn',
                   vmin=-diff_df[metrics].abs().max().max(),
                   vmax= diff_df[metrics].abs().max().max())

    ax.set_xticks(range(len(seasons)))
    ax.set_xticklabels([int(s) for s in seasons], rotation=45)
    ax.set_yticks(range(len(metrics)))
    ax.set_yticklabels(metrics)

    for i, m in enumerate(metrics):
        for j, s in enumerate(seasons):
            val = diff_df.loc[s, m]
            if not np.isnan(val):
                sign = '+' if val > 0 else ''
                label = f'{sign}{val:.1f}%' if m == 'AVG' else f'{sign}{val:.0f}'
                ax.text(j, i, label, ha='center', va='center', fontsize=7.5)

    plt.colorbar(im, ax=ax, label='SSG - 리그평균')
    ax.set_title('SSG vs 리그 평균 (초과/미달)', fontsize=13, fontweight='bold')
    ax.axvline(seasons.index(2022)-0.5 if 2022 in seasons else -1, color='white', linewidth=2)
    plt.tight_layout()
    plt.savefig('../data/ssg_vs_league.png', bbox_inches='tight')
    plt.show()

plot_ssg_vs_league()

In [ ]:
# ── 3. 선수 커리어 트래킹 ─────────────────────────────────────────────────────

def player_career(name: str, highlight_ssg: bool = True):
    """선수명(한글)으로 커리어 스탯 시각화."""
    p = all_df[all_df['선수명'] == name].sort_values('Season')
    if p.empty:
        # 부분 검색
        p = all_df[all_df['선수명'].str.contains(name, na=False)].sort_values('Season')
    if p.empty:
        print(f'"{name}" 선수 없음'); return

    display = p['선수명'].iloc[0]
    seasons = p['Season'].values
    colors  = [SSG_COLOR if t in ('SK','SSG') else '#9E9E9E' for t in p['팀명']]

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle(f'{display} 커리어 트래킹', fontsize=14, fontweight='bold')

    plot_cfg = [
        (axes[0,0], 'AVG',  '타율',   'line'),
        (axes[0,1], 'HR',   '홈런',   'bar'),
        (axes[0,2], 'RBI',  '타점',   'bar'),
        (axes[1,0], 'PA',   '타석',   'bar'),
        (axes[1,1], 'SB',   '도루',   'bar'),
        (axes[1,2], 'BB',   '볼넷',   'bar'),
    ]

    for ax, col, label, kind in plot_cfg:
        if col not in p.columns:
            ax.set_visible(False); continue
        vals = p[col].values
        if kind == 'line':
            ax.plot(seasons, vals, color=SSG_COLOR, linewidth=2.5, marker='o', markersize=7)
            for s, v, c in zip(seasons, vals, colors):
                ax.scatter(s, v, color=c, s=60, zorder=5)
        else:
            ax.bar(seasons, vals, color=colors, alpha=0.85, edgecolor='white')
        ax.set_title(label); ax.set_xlabel('시즌'); ax.grid(alpha=0.3, axis='y')
        ax.tick_params(axis='x', rotation=45)

    # 범례
    ssg_patch = mpatches.Patch(color=SSG_COLOR, label='SSG/SK')
    etc_patch  = mpatches.Patch(color='#9E9E9E', label='타팀')
    fig.legend(handles=[ssg_patch, etc_patch], loc='lower right', fontsize=9)

    plt.tight_layout()
    out = f'../data/{display}_career.png'
    plt.savefig(out, bbox_inches='tight')
    plt.show()
    print(f'저장: {out}')


def search_ssg_players(min_seasons: int = 3):
    """SSG/SK에서 3시즌 이상 뛴 선수 목록"""
    g = ssg_df.groupby('선수명').agg(시즌수=('Season','count'), PA합계=('PA','sum'),
                                     첫시즌=('Season','min'), 마지막시즌=('Season','max'))
    return g[g['시즌수'] >= min_seasons].sort_values('PA합계', ascending=False)

print('=== SSG/SK 장기 재직 선수 (3시즌+) ===')
print(search_ssg_players().head(20).to_string())

In [ ]:
# ── 4. SSG 주요 선수 커리어 (예시) ───────────────────────────────────────────

# SSG/SK 프랜차이즈 대표 선수들
key_players = search_ssg_players(min_seasons=4).head(5).index.tolist()
print('주요 선수:', key_players)

for player in key_players:
    player_career(player)

In [ ]:
# ── 5. 특정 선수 직접 조회 ────────────────────────────────────────────────────
# 원하는 선수 이름으로 바꿔서 실행

player_career('최정')   # 예: 최정, 추신수, 박정권 등

In [ ]:
# ── 6. SSG 시즌별 top 타자 ────────────────────────────────────────────────────

def ssg_season_top(season: int, top_n: int = 10, metric: str = 'AVG'):
    d = ssg_df[(ssg_df['Season'] == season) & (ssg_df['PA'] >= 100)]
    show = ['선수명', 'PA', 'AVG', 'HR', 'RBI', 'SB', 'BB', 'SO', '2B']
    show = [c for c in show if c in d.columns]
    return d.nlargest(top_n, metric)[show]

# 각 시즌 top 5 (AVG 기준)
for season in sorted(ssg_df['Season'].unique())[-5:]:
    print(f'\n=== {int(season)}시즌 SSG top 5 (AVG) ===')
    print(ssg_season_top(int(season), 5).to_string(index=False))